In [6]:
!nvidia-smi

Sun Oct  4 11:30:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
%%writefile matrix_tensorcore.cu

#include <stdio.h>
#include <cuda.h>
#include <cuda_fp16.h>
#include <mma.h>
#include <math.h>

using namespace nvcuda;

#define N 64
#define TILE 16

__global__ void matrixMulTensorCore(const half *A,
                                    const half *B,
                                    float *C)
{
    int tileRow = blockIdx.y;
    int tileCol = blockIdx.x;

    wmma::fragment<wmma::matrix_a, 16, 16, 16,
                   half, wmma::row_major> a_frag;

    wmma::fragment<wmma::matrix_b, 16, 16, 16,
                   half, wmma::row_major> b_frag;

    wmma::fragment<wmma::accumulator, 16, 16, 16,
                   float> c_frag;

    wmma::fill_fragment(c_frag, 0.0f);

    for (int k = 0; k < N; k += TILE)
    {
        const half *A_tile =
            A + tileRow * TILE * N + k;

        const half *B_tile =
            B + k * N + tileCol * TILE;

        wmma::load_matrix_sync(a_frag, A_tile, N);
        wmma::load_matrix_sync(b_frag, B_tile, N);

        wmma::mma_sync(c_frag, a_frag, b_frag, c_frag);
    }

    float *C_tile =
        C + tileRow * TILE * N + tileCol * TILE;

    wmma::store_matrix_sync(C_tile, c_frag, N,
                            wmma::mem_row_major);
}

int main()
{
    half *h_A;
    half *h_B;
    float *h_C;
    float *h_reference;

    h_A = new half[N * N];
    h_B = new half[N * N];
    h_C = new float[N * N];
    h_reference = new float[N * N];

    // Initialize matrices
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            h_A[i * N + j] =
                __float2half((float)((i + j) % 5));

            h_B[i * N + j] =
                __float2half((float)((i - j + 64) % 5));

            h_C[i * N + j] = 0.0f;
            h_reference[i * N + j] = 0.0f;
        }
    }

    // CPU reference
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            float sum = 0.0f;

            for (int k = 0; k < N; k++)
            {
                sum += __half2float(h_A[i * N + k]) *
                       __half2float(h_B[k * N + j]);
            }

            h_reference[i * N + j] = sum;
        }
    }

    half *d_A;
    half *d_B;
    float *d_C;

    cudaMalloc(&d_A, N * N * sizeof(half));
    cudaMalloc(&d_B, N * N * sizeof(half));
    cudaMalloc(&d_C, N * N * sizeof(float));

    cudaMemcpy(d_A, h_A,
               N * N * sizeof(half),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, h_B,
               N * N * sizeof(half),
               cudaMemcpyHostToDevice);

    cudaMemset(d_C, 0, N * N * sizeof(float));

    // 64x64 / 16x16 = 4x4 blocks = 16 tiles
    dim3 blockDim(32);
    dim3 gridDim(N / TILE, N / TILE);

    matrixMulTensorCore<<<gridDim, blockDim>>>(d_A, d_B, d_C);

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess)
    {
        printf("Kernel launch error: %s\n",
               cudaGetErrorString(error));
        return 1;
    }

    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C,
               N * N * sizeof(float),
               cudaMemcpyDeviceToHost);

    // Verify
    bool correct = true;
    float maxError = 0.0f;

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            float error =
                fabs(h_C[i * N + j] -
                     h_reference[i * N + j]);

            if (error > maxError)
                maxError = error;

            if (error > 0.1f)
                correct = false;
        }
    }

    printf("\n========================================\n");
    printf("Tensor Core Matrix Multiplication\n");
    printf("========================================\n");

    printf("Matrix A : 64 x 64\n");
    printf("Matrix B : 64 x 64\n");
    printf("Matrix C : 64 x 64\n");
    printf("Tile     : 16 x 16\n");
    printf("Tiles    : 4 x 4 = 16\n");
    printf("K tiles  : 4\n");
    printf("Max error: %f\n", maxError);

    if (correct)
        printf("Result   : CORRECT\n");
    else
        printf("Result   : INCORRECT\n");

    printf("\nFirst 4x4 elements of C:\n");

    for (int i = 0; i < 4; i++)
    {
        for (int j = 0; j < 4; j++)
        {
            printf("%8.2f ", h_C[i * N + j]);
        }
        printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;
    delete[] h_reference;

    return 0;
}

Writing matrix_tensorcore.cu


In [8]:
!nvcc -arch=sm_75 matrix_tensorcore.cu -o matrix_tensorcore

In [9]:
!./matrix_tensorcore


Tensor Core Matrix Multiplication
Matrix A : 64 x 64
Matrix B : 64 x 64
Matrix C : 64 x 64
Tile     : 16 x 16
Tiles    : 4 x 4 = 16
K tiles  : 4
Max error: 0.000000
Result   : CORRECT

First 4x4 elements of C:
  248.00   187.00   191.00   260.00 
  195.00   195.00   260.00   390.00 
  192.00   258.00   389.00   260.00 
  254.00   386.00   258.00   195.00 
